# Using StatsForecast + CrostonClassic for forecasting

In [1]:
import pandas as pd
import numpy as np 

In [2]:
sales_orders = pd.read_csv(r"D:\Artificial Intelligence\SupplyChainManagementProject\supply-chain-ai\data\raw\sales_orders.csv")

In [3]:
sales_orders.head()

,Order_ID,Customer_ID,Product_ID,Order_Date,Order_Status,Order_Quantity,Unit_Price,Discount,Shipping_Mode,Shipping_Carrier,Shipping_Date_Scheduled,Shipping_Date_Actual,Delivery_Status,Late_Delivery_Risk_Flag,VAT_Rate,COGS,Unit_Price_Effective,Order_Total,VAT_Amount,Profit_Per_Order
0,ORD0000001,CUST00481,PROD00135,2023-06-16,Completed,8,200.54,0.12,Standard,CarrierA,2023-06-21,2023-06-21,On Time,0,0.19,1510.96,176.4752,1411.8016,268.242304,-99.1584
1,ORD0000002,CUST00111,PROD00147,2023-09-23,Pending,3,220.96,0.18,Same Day,CarrierB,2023-09-26,2023-09-26,On Time,0,0.20,371.61,181.1872,543.5616,108.712320,171.9516
2,ORD0000003,CUST00104,PROD00172,2022-12-17,Cancelled,2,15.87,0.09,Economy,CarrierD,2022-12-18,2022-12-20,Late,1,0.20,16.46,14.4417,28.8834,5.776680,12.4234
3,ORD0000004,CUST00316,PROD00063,2023-10-11,Pending,7,191.91,0.06,Express,CarrierC,2023-10-15,2023-10-19,Late,1,0.00,1166.20,180.3954,1262.7678,0.000000,96.5678
4,ORD0000005,CUST00171,PROD00065,2022-10-04,Completed,6,192.00,0.19,Same Day,CarrierB,2022-10-05,2022-10-05,On Time,0,0.20,358.68,155.5200,933.1200,186.624000,574.4400


In [4]:
df = sales_orders[sales_orders['Order_Status'] == 'Completed'][['Product_ID','Order_Date','Order_Quantity']].copy()

In [5]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

In [6]:
df.head()

,Product_ID,Order_Date,Order_Quantity
0,PROD00135,2023-06-16,8
4,PROD00065,2022-10-04,6
5,PROD00039,2022-09-12,6
6,PROD00109,2022-09-21,5
7,PROD00026,2023-11-10,12


### Converting Daily data to Weekly aggregated data

In [7]:
weekly_df = (
    df.groupby(
        ['Product_ID', pd.Grouper(key='Order_Date', freq='W')]
    )['Order_Quantity']
    .sum()
    .reset_index()
)

In [8]:
weekly_df.head()

,Product_ID,Order_Date,Order_Quantity
0,PROD00001,2022-01-09,3
1,PROD00001,2022-01-30,7
2,PROD00001,2022-02-06,10
3,PROD00001,2022-02-27,6
4,PROD00001,2022-03-06,3


In [9]:
result = []

for product, group in df.groupby('Product_ID'):

    # set date index
    group = group.set_index('Order_Date')

    # create continuous weekly dates
    group = group.resample('W').sum()

    # fill missing demand with 0
    group['Order_Quantity'] = group['Order_Quantity'].fillna(0)

    # restore product id
    group['Product_ID'] = product

    # reset index
    group = group.reset_index()

    result.append(group)

weekly_df = pd.concat(result)

In [10]:
weekly_df = weekly_df.rename(columns={
    'Product_ID': 'unique_id',
    'Order_Date': 'ds',
    'Order_Quantity': 'y'
})

## Splitting train and test data based on date's 

- In TimeSeries Forecasting the dataset need to follow sequential order
- last 12 weeks of data is splitted for testing and the remaining data is for training 

In [11]:
cutoff = weekly_df['ds'].max() - pd.Timedelta(weeks=12)

train = weekly_df[weekly_df['ds'] <= cutoff]
test =weekly_df[weekly_df['ds'] > cutoff]

In [12]:
cutoff

Timestamp('2024-07-07 00:00:00')

In [13]:
weekly_df.head()

,ds,unique_id,y
0,2022-01-09,PROD00001,3
1,2022-01-16,PROD00001,0
2,2022-01-23,PROD00001,0
3,2022-01-30,PROD00001,7
4,2022-02-06,PROD00001,10


In [14]:
train.head()

,ds,unique_id,y
0,2022-01-09,PROD00001,3
1,2022-01-16,PROD00001,0
2,2022-01-23,PROD00001,0
3,2022-01-30,PROD00001,7
4,2022-02-06,PROD00001,10


In [15]:
test.head()

,ds,unique_id,y
131,2024-07-14,PROD00001,7
132,2024-07-21,PROD00001,5
133,2024-07-28,PROD00001,5
134,2024-08-04,PROD00001,0
135,2024-08-11,PROD00001,0


## Croston algorithm 

- It is sepecificaly designed for intermittent type of, it hadles datasets with many zeros and sudden spikes

In [16]:
from statsforecast import StatsForecast
from statsforecast.models import CrostonClassic, SeasonalNaive

D:\Artificial Intelligence\SupplyChainManagementProject\project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
horizon = test.groupby('unique_id').size().max()
horizon

np.int64(12)

In [18]:
models = [CrostonClassic()]
freq = 'W'

In [19]:
sf = StatsForecast(models=models, freq=freq)

In [20]:
sf.fit(df=train)

StatsForecast(models=[CrostonClassic])

## this line return the fitted models

In [21]:
result = sf.fitted_[0,0].model_

In [22]:
result

{'mean': array([3.77813587]),
 'fitted': array([      nan, 3.       , 3.       , 3.       , 2.8333333, 3.4406776,
        3.4406776, 3.4406776, 3.1233475, 3.114044 , 3.1825519, 3.4845896,
        3.5262384, 3.7300487, 3.7300487, 3.7300487, 3.8600311, 4.0181794,
        4.0181794, 3.662963 , 3.662963 , 3.662963 , 3.662963 , 3.662963 ,
        2.9695792, 3.265758 , 3.3722448, 3.5400736, 3.5400736, 3.5350926,
        3.4359422, 4.0093937, 4.0093937, 4.0093937, 4.0093937, 4.0093937,
        3.4103436, 3.4103436, 3.4103436, 3.4103436, 3.4103436, 3.4103436,
        3.035862 , 3.3153822, 3.6957457, 3.6957457, 3.6763487, 3.6763487,
        3.6763487, 3.2497404, 3.2497404, 3.2497404, 3.2497404, 3.032849 ,
        3.032849 , 3.032849 , 3.032849 , 3.032849 , 3.032849 , 3.032849 ,
        2.3757582, 2.399796 , 2.5475137, 2.5475137, 2.4599838, 2.7944791,
        2.8511102, 2.8511102, 2.8511102, 2.8511102, 2.7904658, 2.7904658,
        2.6774786, 2.6774786, 2.750402 , 2.9029467, 2.9029467, 2.9121146

## This is the simple version of fitting and predicting the model

In [23]:
yhat = sf.forecast(df=train,h=horizon)
yhat

,unique_id,ds,CrostonClassic
0,PROD00001,2024-07-14,3.778136
1,PROD00001,2024-07-21,3.778136
2,PROD00001,2024-07-28,3.778136
3,PROD00001,2024-08-04,3.778136
4,PROD00001,2024-08-11,3.778136
...,...,...,...
2395,PROD00200,2024-09-01,2.853414
2396,PROD00200,2024-09-08,2.853414
2397,PROD00200,2024-09-15,2.853414
2398,PROD00200,2024-09-22,2.853414


In [24]:
forecast_df = sf.predict(h=horizon)
forecast_df

,unique_id,ds,CrostonClassic
0,PROD00001,2024-07-14,3.778136
1,PROD00001,2024-07-21,3.778136
2,PROD00001,2024-07-28,3.778136
3,PROD00001,2024-08-04,3.778136
4,PROD00001,2024-08-11,3.778136
...,...,...,...
2395,PROD00200,2024-09-01,2.853414
2396,PROD00200,2024-09-08,2.853414
2397,PROD00200,2024-09-15,2.853414
2398,PROD00200,2024-09-22,2.853414


In [25]:
eval_df = test.merge(
    forecast_df,
    on=['unique_id', 'ds'],
    how='left'
)

In [26]:
eval_df.head()

,ds,unique_id,y,CrostonClassic
0,2024-07-14,PROD00001,7,3.778136
1,2024-07-21,PROD00001,5,3.778136
2,2024-07-28,PROD00001,5,3.778136
3,2024-08-04,PROD00001,0,3.778136
4,2024-08-11,PROD00001,0,3.778136


In [27]:
from functools import partial

import utilsforecast.losses as ufl
from utilsforecast.evaluation import evaluate

In [28]:
metrics = evaluate(

    eval_df,

    metrics=[
        ufl.mae,
        ufl.rmse,
        ufl.smape,

        # scaled error
        partial(
            ufl.mase,
            seasonality=1
        )
    ],

    train_df=train
)
metrics

,unique_id,metric,CrostonClassic
0,PROD00001,mae,3.740621
1,PROD00002,mae,2.875000
2,PROD00003,mae,2.409050
3,PROD00004,mae,2.905321
4,PROD00005,mae,3.791199
...,...,...,...
795,PROD00196,mase,0.998345
796,PROD00197,mase,0.586263
797,PROD00198,mase,0.550130
798,PROD00199,mase,0.681501


In [29]:
summary_metrics = (
    metrics.groupby('metric')
    ['CrostonClassic']
    .mean()
)

print("\nAVERAGE METRICS:\n")
print(summary_metrics)


AVERAGE METRICS:

metric
mae      3.423023
mase     0.845115
rmse     4.154774
smape    0.656360
Name: CrostonClassic, dtype: float64


# Forecast Evaluation Metrics

---

## 1. Mean Absolute Error (MAE)

### Formula

$$
\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \left| y_i - \hat{y}_i \right|
$$

Where:
- $y_i$ = actual demand value
- $\hat{y}_i$ = forecasted value
- $n$ = number of observations

---

### Interpretation

MAE measures the average absolute forecasting error.

**Example:** $\text{MAE} = 3.4$

> On average, the model prediction is off by approximately 3.4 units.

---

### Business Meaning

Lower MAE indicates:
- better forecast accuracy
- more reliable inventory planning
- reduced procurement error

MAE is one of the most interpretable forecasting metrics because it directly represents unit-level forecasting error.

---

## 2. Root Mean Squared Error (RMSE)

### Formula

$$
\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2}
$$

---

### Interpretation

RMSE penalizes larger forecast errors more heavily because errors are squared before averaging.

**Example:** $\text{RMSE} = 4.1$

> The model occasionally produces larger forecasting errors.

---

### Business Meaning

Lower RMSE indicates:
- fewer severe forecast failures
- better handling of demand spikes
- more stable forecasting behavior

RMSE is highly sensitive to outliers and extreme forecasting misses.

---

## 3. Symmetric Mean Absolute Percentage Error (SMAPE)

### Formula

$$
\text{SMAPE} = \frac{1}{n} \sum_{i=1}^{n} \frac{2\,\left| y_i - \hat{y}_i \right|}{\left| y_i \right| + \left| \hat{y}_i \right|}
$$

---

### Interpretation

SMAPE measures percentage-based forecasting accuracy while reducing issues caused by zero values.

**Example:** $\text{SMAPE} = 0.65$

> The average relative forecasting error is approximately 65%.

---

### Business Meaning

Lower SMAPE indicates:
- improved proportional forecasting accuracy
- better relative demand estimation

SMAPE is preferred over MAPE for intermittent demand datasets because it handles zeros more safely.

---

## 4. Mean Absolute Scaled Error (MASE)

### Formula

$$
\text{MASE} = \frac{\text{MAE}_{\text{model}}}{\text{MAE}_{\text{naive}}}
$$

Where the naive forecast MAE is defined as:

$$
\text{MAE}_{\text{naive}} = \frac{1}{n-1} \sum_{i=2}^{n} \left| y_i - y_{i-1} \right|
$$

---

### Interpretation

MASE compares the forecasting model against a naive baseline forecasting approach.

**Example:** $\text{MASE} = 0.84$

> The forecasting model performs better than naive forecasting.

---

### MASE Interpretation Table

| MASE Value | Interpretation |
|---|---|
| $< 1$ | Better than naive forecasting |
| $= 1$ | Same as naive forecasting |
| $> 1$ | Worse than naive forecasting |

---

### Business Meaning

MASE is highly useful for intermittent demand forecasting because:
- it is scale independent
- works well across multiple SKUs
- compares against a simple baseline

A MASE score below 1 generally indicates strong forecasting performance.

---

## Metric Priority for Intermittent Demand Forecasting

| Metric | Importance |
|---|---|
| MAE | High |
| MASE | High |
| RMSE | Medium |
| SMAPE | Medium |

---

## Important Note on MAPE

MAPE is **not recommended** for intermittent demand forecasting. Its formula,

$$
\text{MAPE} = \frac{1}{n} \sum_{i=1}^{n} \left| \frac{y_i - \hat{y}_i}{y_i} \right|,
$$

produces division-by-zero instability whenever $y_i = 0$, which is common in intermittent demand datasets, leading to misleading percentage errors.

In [30]:
forecast_df.head()

,unique_id,ds,CrostonClassic
0,PROD00001,2024-07-14,3.778136
1,PROD00001,2024-07-21,3.778136
2,PROD00001,2024-07-28,3.778136
3,PROD00001,2024-08-04,3.778136
4,PROD00001,2024-08-11,3.778136


In [31]:
test.head(20)

,ds,unique_id,y
131,2024-07-14,PROD00001,7
132,2024-07-21,PROD00001,5
133,2024-07-28,PROD00001,5
134,2024-08-04,PROD00001,0
135,2024-08-11,PROD00001,0
136,2024-08-18,PROD00001,13
130,2024-07-14,PROD00002,0
131,2024-07-21,PROD00002,5
132,2024-07-28,PROD00002,4
133,2024-08-04,PROD00002,0


In [32]:
eval_df = test.merge(
    forecast_df,
    on=['unique_id', 'ds'],
    how='left'
)

In [33]:
eval_df['error'] = (
    eval_df['y']
    - eval_df['CrostonClassic']
)

In [34]:
eval_df['abs_error'] = (
    eval_df['error']
    .abs()
)

In [35]:
eval_df

,ds,unique_id,y,CrostonClassic,error,abs_error
0,2024-07-14,PROD00001,7,3.778136,3.221864,3.221864
1,2024-07-21,PROD00001,5,3.778136,1.221864,1.221864
2,2024-07-28,PROD00001,5,3.778136,1.221864,1.221864
3,2024-08-04,PROD00001,0,3.778136,-3.778136,3.778136
4,2024-08-11,PROD00001,0,3.778136,-3.778136,3.778136
...,...,...,...,...,...,...
2038,2024-09-01,PROD00200,11,2.853414,8.146586,8.146586
2039,2024-09-08,PROD00200,0,2.853414,-2.853414,2.853414
2040,2024-09-15,PROD00200,0,2.853414,-2.853414,2.853414
2041,2024-09-22,PROD00200,0,2.853414,-2.853414,2.853414


In [36]:
prods = eval_df[['unique_id','y','CrostonClassic']].groupby(by='unique_id').sum()

In [37]:
prods.head(10)

,y,CrostonClassic
unique_id,,
PROD00001,30,22.668816
PROD00002,23,19.460455
PROD00003,19,43.213577
PROD00004,29,32.543892
PROD00005,39,31.483160
PROD00006,33,28.029711
PROD00007,61,33.756538
PROD00008,32,14.573690
PROD00009,47,23.719547


## Saving the model 

In [38]:
import pickle

In [41]:
with open('croston_model.pkl','wb') as file:
    pickle.dump(sf,file)

## Loading the model and testing

In [42]:
with open('croston_model.pkl','rb') as file:
    model = pickle.load(file)

In [43]:
model

StatsForecast(models=[CrostonClassic])

In [44]:
result = model.predict(horizon)

In [45]:
result

,unique_id,ds,CrostonClassic
0,PROD00001,2024-07-14,3.778136
1,PROD00001,2024-07-21,3.778136
2,PROD00001,2024-07-28,3.778136
3,PROD00001,2024-08-04,3.778136
4,PROD00001,2024-08-11,3.778136
...,...,...,...
2395,PROD00200,2024-09-01,2.853414
2396,PROD00200,2024-09-08,2.853414
2397,PROD00200,2024-09-15,2.853414
2398,PROD00200,2024-09-22,2.853414


In [47]:
result['CrostonClassic'].describe()

count    2400.000000
mean        3.025098
std         0.662118
min         1.324881
25%         2.555744
50%         2.932917
75%         3.415490
max         4.996274
Name: CrostonClassic, dtype: float64